# Module 10: Hybrid Feature Selection Pipeline

## Learning Objectives
By completing this notebook, you will:
1. Build a three-stage filter → embedded → GA cascade pipeline
2. Measure and compare computational time with and without filter pre-screening
3. Compare final feature set quality between cascaded and non-cascaded approaches
4. Visualise the feature funnel: how many features survive each stage
5. Understand where time is actually spent in a hybrid pipeline

## Prerequisites
- Notebooks 01 (ensemble selection) and the hybrid methods guide
- Familiarity with LASSO, genetic algorithms, and cross-validation

## Estimated Time: 15 minutes

## Dataset
We use the **MADELON** artificial dataset from OpenML (dataset ID 1485). MADELON has 500 features, 2600 samples, and only 20 informative features — making it an ideal stress test for hybrid selection. The remaining 480 features are noise or distractor features.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
import random
from typing import Callable

from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

warnings.filterwarnings('ignore')
np.random.seed(42)

# --- Load dataset ---
# We use sklearn's make_classification to generate a MADELON-like dataset
# This ensures reproducibility without requiring network access to OpenML
print("Generating high-dimensional dataset (500 features, 2000 samples)...")
X_raw, y_raw = make_classification(
    n_samples=2000,
    n_features=500,
    n_informative=20,     # 20 truly informative features
    n_redundant=10,       # 10 redundant combinations of informative features
    n_repeated=0,
    n_clusters_per_class=2,
    flip_y=0.05,          # 5% label noise
    random_state=42
)

feature_names = [f'X{i:03d}' for i in range(X_raw.shape[1])]
X = pd.DataFrame(X_raw, columns=feature_names)
y = y_raw

print(f"Dataset shape: {X.shape}")
print(f"Class balance: {y.mean():.3f}")
print(f"Truly informative features: 20 / {X.shape[1]}")
print(f"Noise features: {X.shape[1] - 20} / {X.shape[1]}")

## 1. Stage 1: Filter Pre-Screening

The filter stage scores all 500 features using Mutual Information with the target. Features with MI below a threshold are discarded. We time this carefully to establish the baseline cost.

In [ ]:
def filter_stage(X: pd.DataFrame, y: np.ndarray,
                  top_k: int, method: str = 'mi') -> pd.DataFrame:
    """
    Filter stage: score all features and retain the top-k.

    Parameters
    ----------
    X : full feature matrix
    y : target
    top_k : number of features to keep
    method : 'mi' (mutual information) or 'variance'

    Returns
    -------
    Reduced DataFrame and array of MI scores.
    """
    if method == 'mi':
        scores = mutual_info_classif(X.values, y, random_state=42)
    elif method == 'variance':
        scores = X.var().values
    else:
        raise ValueError(f'Unknown filter method: {method}')

    # Keep top-k features by score
    top_indices = np.argsort(-scores)[:top_k]
    selected_cols = X.columns[top_indices]
    return X[selected_cols].copy(), scores


FILTER_TOP_K = 100   # Keep top 100 features at the filter stage

t_filter_start = time.perf_counter()
X_filtered, mi_scores = filter_stage(X, y, top_k=FILTER_TOP_K)
t_filter = time.perf_counter() - t_filter_start

print(f"Filter stage: {X.shape[1]} → {X_filtered.shape[1]} features in {t_filter:.2f}s")

# Visualise the MI score distribution
fig, ax = plt.subplots(figsize=(12, 4))
sorted_mi = np.sort(mi_scores)[::-1]
colours = ['#4CAF50' if i < FILTER_TOP_K else '#F44336' for i in range(len(sorted_mi))]
ax.bar(range(len(sorted_mi)), sorted_mi, color=colours, width=1.0)
ax.axvline(FILTER_TOP_K - 0.5, color='black', linewidth=2, linestyle='--',
           label=f'Filter cutoff (top-{FILTER_TOP_K})')
ax.set_xlabel('Feature rank by MI score', fontsize=11)
ax.set_ylabel('Mutual Information with target', fontsize=11)
ax.set_title(f'MI Score Distribution: 500 Features\n(Green = kept, Red = discarded by filter)',
             fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('mi_score_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Stage 2: Embedded Refinement (LASSO)

LASSO on the 100-feature pre-screened set identifies a sparse linear model. Features with non-zero coefficients advance to stage 3.

In [ ]:
def embedded_stage_lasso(X_filtered: pd.DataFrame, y: np.ndarray,
                           top_k: int = 30) -> pd.DataFrame:
    """
    LASSO embedded stage on pre-filtered features.

    Fits LassoCV (cross-validated lambda), selects features with
    |coefficient| >= threshold (or top-k if LASSO is too aggressive).

    Returns
    -------
    Reduced DataFrame and coefficient array.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_filtered)

    lasso = LassoCV(cv=5, random_state=42, max_iter=10000, n_alphas=50)
    lasso.fit(X_scaled, y)

    coef_abs = np.abs(lasso.coef_)

    # Non-zero coefficients
    nonzero_mask = coef_abs > 0
    n_nonzero = nonzero_mask.sum()

    if n_nonzero >= top_k:
        # Enough non-zero: keep top-k by magnitude
        threshold = np.sort(coef_abs)[::-1][top_k - 1]
        mask = coef_abs >= threshold
    elif n_nonzero >= 5:
        # LASSO selected a reasonable number: keep all non-zero
        mask = nonzero_mask
    else:
        # LASSO too aggressive: keep top-k by magnitude
        threshold = np.sort(coef_abs)[::-1][min(top_k, len(coef_abs)) - 1]
        mask = coef_abs >= threshold

    selected_cols = X_filtered.columns[mask]
    print(f"  LASSO: optimal alpha={lasso.alpha_:.4f}, "
          f"non-zero coefficients={n_nonzero}, selected={mask.sum()}")

    return X_filtered[selected_cols].copy(), coef_abs


EMBEDDED_TOP_K = 35

t_embedded_start = time.perf_counter()
X_embedded, lasso_coef = embedded_stage_lasso(X_filtered, y, top_k=EMBEDDED_TOP_K)
t_embedded = time.perf_counter() - t_embedded_start

print(f"\nEmbedded stage: {X_filtered.shape[1]} → {X_embedded.shape[1]} features in {t_embedded:.2f}s")

## 3. Stage 3: GA Wrapper on Reduced Candidate Set

The GA wrapper operates on the ~35 candidates from stage 2. Because the search space is now small, the GA converges quickly. We use seeded initialisation (pre-seeding with top-ranked candidates) for faster convergence.

In [ ]:
def ga_wrapper(X_candidates: pd.DataFrame, y: np.ndarray,
                pop_size: int = 25,
                n_generations: int = 40,
                mutation_rate: float = 0.08,
                target_features: int | None = None,
                random_state: int = 42) -> tuple:
    """
    Binary-encoded GA wrapper operating on a pre-screened feature subset.

    Fitness: 5-fold cross-validated balanced accuracy of GradientBoosting.

    Returns
    -------
    (selected_feature_names, best_fitness, fitness_history)
    """
    rng = random.Random(random_state)
    p = X_candidates.shape[1]
    X_arr = X_candidates.values
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    def fitness(chrom: list) -> float:
        selected = [i for i, g in enumerate(chrom) if g == 1]
        if not selected:
            return 0.0
        # Penalise using too many features (encourages parsimony)
        n_sel = len(selected)
        if target_features and n_sel > target_features * 2:
            return 0.0
        model = GradientBoostingClassifier(n_estimators=50, random_state=random_state)
        scores = cross_val_score(model, X_arr[:, selected], y,
                                  cv=cv, scoring='balanced_accuracy')
        return float(scores.mean())

    def make_chrom():
        # Seeded: first features (highest-LASSO-scored) more likely to be 1
        chrom = [1 if rng.random() < max(0.7 - i * 0.02, 0.2) else 0
                  for i in range(p)]
        if sum(chrom) == 0:
            chrom[0] = 1
        return chrom

    population = [make_chrom() for _ in range(pop_size)]
    best_fitness_history = []

    for gen in range(n_generations):
        scored = [(fitness(c), c) for c in population]
        scored.sort(key=lambda x: x[0], reverse=True)
        best_fitness_history.append(scored[0][0])

        n_elite = max(2, pop_size // 5)
        new_pop = [c for _, c in scored[:n_elite]]

        while len(new_pop) < pop_size:
            # Tournament selection from top half
            half = scored[:pop_size // 2 + 1]
            p1 = rng.choice(half)[1]
            p2 = rng.choice(half)[1]
            cut = rng.randint(1, p - 1)
            child = p1[:cut] + p2[cut:]
            child = [1 - g if rng.random() < mutation_rate else g for g in child]
            if sum(child) == 0:
                child[rng.randint(0, p - 1)] = 1
            new_pop.append(child)

        population = new_pop[:pop_size]

    # Get best chromosome
    final_scored = [(fitness(c), c) for c in population]
    best_chrom = max(final_scored, key=lambda x: x[0])[1]
    selected_names = X_candidates.columns[[i for i, g in enumerate(best_chrom) if g == 1]].tolist()
    best_fit = max(final_scored, key=lambda x: x[0])[0]

    return selected_names, best_fit, best_fitness_history


TARGET_FEATURES = 20
print(f"Running GA wrapper on {X_embedded.shape[1]} candidate features...")
print(f"(Target: {TARGET_FEATURES} final features)")

t_ga_start = time.perf_counter()
final_features, best_fitness, fitness_history = ga_wrapper(
    X_embedded, y,
    pop_size=25,
    n_generations=40,
    target_features=TARGET_FEATURES
)
t_ga = time.perf_counter() - t_ga_start

print(f"\nGA wrapper: {X_embedded.shape[1]} → {len(final_features)} features in {t_ga:.2f}s")
print(f"Best CV fitness: {best_fitness:.4f}")
print(f"Final features: {final_features}")

## 4. Timing Comparison: With vs Without Pre-Screening

Now we benchmark the GA running directly on all 500 features (no pre-screening) versus the cascaded pipeline. We use the same GA settings but applied to the full feature space.

In [ ]:
# Run GA directly on all 500 features (no pre-screening)
# Note: we use fewer generations to keep runtime manageable for demonstration
# In production, you'd use the same settings — the timing difference is even larger

print("Running GA directly on all 500 features (no pre-screening)...")
print("(Using 15 generations to keep demo runtime reasonable; full run would be ~10× longer)")

t_ga_full_start = time.perf_counter()
full_features, full_fitness, full_history = ga_wrapper(
    X,  # full 500-feature dataset
    y,
    pop_size=25,
    n_generations=15,   # fewer gens for demo
    target_features=TARGET_FEATURES
)
t_ga_full = time.perf_counter() - t_ga_full_start

print(f"\nGA on full 500 features: {len(full_features)} features selected in {t_ga_full:.2f}s")
print(f"Best CV fitness (15 gen): {full_fitness:.4f}")

# Summary timing
print("\n" + "=" * 55)
print("TIMING SUMMARY")
print("=" * 55)
print(f"{'Stage':30s} {'Time (s)':>10s} {'Features':>10s}")
print("-" * 55)
print(f"{'Filter (MI)':30s} {t_filter:10.2f} {X.shape[1]} → {X_filtered.shape[1]}")
print(f"{'Embedded (LASSO)':30s} {t_embedded:10.2f} {X_filtered.shape[1]} → {X_embedded.shape[1]}")
print(f"{'GA wrapper (on reduced set)':30s} {t_ga:10.2f} {X_embedded.shape[1]} → {len(final_features)}")
pipeline_total = t_filter + t_embedded + t_ga
print("-" * 55)
print(f"{'CASCADED TOTAL':30s} {pipeline_total:10.2f} {X.shape[1]} → {len(final_features)}")
print()
print(f"{'GA on full 500 features (15 gen)':30s} {t_ga_full:10.2f} {X.shape[1]} → {len(full_features)}")
# Extrapolate: full 40 generations would take 40/15 × t_ga_full
t_ga_full_extrap = t_ga_full * (40 / 15)
print(f"{'GA on full (40 gen, extrapolated)':30s} {t_ga_full_extrap:10.2f}")
print()
print(f"Estimated speedup (cascade vs full GA): {t_ga_full_extrap / pipeline_total:.1f}×")

## 5. Feature Funnel Visualisation

Visualise how many features survive each stage of the cascade pipeline.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Plot 1: Feature funnel ---
stages = ['Original\n500', 'After Filter\n100', 'After LASSO\n~35', f'Final\n{len(final_features)}']
counts = [500, X_filtered.shape[1], X_embedded.shape[1], len(final_features)]
times_per_stage = [t_filter, t_embedded, t_ga, 0]

colours = ['#F44336', '#FF9800', '#FFC107', '#4CAF50']
bars = axes[0].bar(stages, counts, color=colours, edgecolor='black', linewidth=0.8,
                    width=0.6)
for bar, count, t in zip(bars, counts, times_per_stage):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                  bar.get_height() + 5,
                  f'{count}\n({t:.1f}s)' if t > 0 else f'{count}',
                  ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Number of features', fontsize=11)
axes[0].set_title('Feature Funnel: Cascade Pipeline\n(Each bar shows features remaining + stage time)',
                   fontsize=11)
axes[0].set_ylim(0, 580)

# Annotate with reduction percentages
for i in range(1, len(counts)):
    reduction = (1 - counts[i] / counts[i-1]) * 100
    axes[0].annotate('', xy=(i, counts[i] + 15), xytext=(i-1, counts[i-1] - 15),
                      arrowprops=dict(arrowstyle='->', color='gray', lw=2))
    mid_x = i - 0.5
    mid_y = (counts[i] + counts[i-1]) / 2
    axes[0].text(mid_x, mid_y, f'{reduction:.0f}%\nremoved',
                  ha='center', va='center', fontsize=9, color='gray',
                  bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

# --- Plot 2: GA convergence comparison ---
# Pad short history with last value for comparison
max_gen = max(len(fitness_history), len(full_history))
padded_cascade = fitness_history + [fitness_history[-1]] * (max_gen - len(fitness_history))
padded_full = full_history + [full_history[-1]] * (max_gen - len(full_history))

axes[1].plot(range(1, len(fitness_history) + 1), fitness_history,
              label=f'Cascade GA ({X_embedded.shape[1]} features)', color='#4CAF50', linewidth=2)
axes[1].plot(range(1, len(full_history) + 1), full_history,
              label=f'Full GA ({X.shape[1]} features, 15 gen)', color='#F44336',
              linewidth=2, linestyle='--')
axes[1].set_xlabel('Generation', fontsize=11)
axes[1].set_ylabel('Best CV balanced accuracy', fontsize=11)
axes[1].set_title('GA Convergence: Cascade vs Direct Search\n(Cascade starts from better candidate set)',
                   fontsize=11)
axes[1].legend(fontsize=10)
axes[1].set_ylim(0.5, 1.0)

plt.tight_layout()
plt.savefig('feature_funnel.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Quality Comparison: Cascaded vs Direct

Compare the downstream model performance of:
- Features selected by the cascade pipeline
- Features selected by GA directly on all 500 features
- Top-20 by MI filter alone (no downstream refinement)
- All 500 features (baseline)

In [ ]:
eval_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get column indices for each approach
final_indices = [list(X.columns).index(f) for f in final_features]
full_indices = [list(X.columns).index(f) for f in full_features]
mi_top20_indices = np.argsort(-mi_scores)[:20].tolist()

approaches = {
    f'Cascade pipeline ({len(final_features)} features)': final_indices,
    f'Direct GA on 500 ({len(full_features)} features)': full_indices,
    'MI filter top-20 only': mi_top20_indices,
    'All 500 features': list(range(X.shape[1])),
}

print("Downstream model performance (5-fold CV balanced accuracy):")
print(f"{'Approach':45s} {'Mean':>8s} {'Std':>8s} {'N features':>12s}")
print("-" * 78)

results = {}
for label, indices in approaches.items():
    X_sub = X.values[:, indices]
    scores = cross_val_score(eval_model, X_sub, y, cv=cv, scoring='balanced_accuracy')
    results[label] = (scores.mean(), scores.std(), len(indices))
    print(f"{label:45s} {scores.mean():8.4f} {scores.std():8.4f} {len(indices):12d}")

## 7. Timing Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Time breakdown for cascade ---
stage_names = ['MI Filter', 'LASSO\nEmbedded', f'GA Wrapper\n({X_embedded.shape[1]} features)']
stage_times = [t_filter, t_embedded, t_ga]
stage_colours = ['#42A5F5', '#66BB6A', '#FFA726']

wedges, texts, autotexts = axes[0].pie(
    stage_times,
    labels=stage_names,
    colors=stage_colours,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.8
)
for at in autotexts:
    at.set_fontsize(10)
axes[0].set_title(f'Time Breakdown: Cascade Pipeline\nTotal = {pipeline_total:.1f}s', fontsize=11)

# Add time labels
for i, (stage, t) in enumerate(zip(stage_names, stage_times)):
    texts[i].set_text(f'{stage}\n({t:.1f}s)')

# --- Plot 2: Quality vs Time trade-off ---
labels = list(results.keys())
means = [results[l][0] for l in labels]
n_features = [results[l][2] for l in labels]
times_approx = [pipeline_total, t_ga_full_extrap,
                 t_filter,  # filter only
                 0.01]  # all features (no selection)

scatter_colours = ['#4CAF50', '#F44336', '#FF9800', '#9E9E9E']
for i, (label, mean, t) in enumerate(zip(labels, means, times_approx)):
    axes[1].scatter(t, mean, s=200, color=scatter_colours[i],
                     zorder=5, edgecolor='black', linewidth=1.2)
    n_feat_label = f'{n_features[i]} feat'
    axes[1].annotate(f'{label.split("(")[0].strip()}\n{n_feat_label}',
                      xy=(t, mean), xytext=(10, 5),
                      textcoords='offset points', fontsize=8,
                      bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

axes[1].set_xlabel('Computation time (seconds)', fontsize=11)
axes[1].set_ylabel('CV Balanced Accuracy', fontsize=11)
axes[1].set_title('Quality vs Computation Trade-off\n(Top-left corner = ideal)', fontsize=11)
axes[1].set_xlim(-1, max(times_approx) * 1.2)

plt.tight_layout()
plt.savefig('timing_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Exercise 8.1: Vary Filter Aggressiveness

**Task:** Run the cascade pipeline with three different filter sizes: `top_k = 50`, `100`, and `200`. For each:
- Record the total cascade time
- Record the final CV accuracy
- Plot the trade-off between filter size, time, and accuracy

**Question to answer:** Is there a point of diminishing returns where a larger filter top_k no longer improves quality but only increases time?

In [ ]:
# YOUR CODE HERE
# ---------------
# For filter_k in [50, 100, 200]:
#   1. Run filter_stage with top_k = filter_k
#   2. Run embedded_stage_lasso with top_k = min(35, filter_k // 3)
#   3. Run ga_wrapper with n_generations=20
#   4. Evaluate final features with cross_val_score
#   5. Record timing and accuracy

filter_sizes = [50, 100, 200]
filter_comparison = []

for filter_k in filter_sizes:
    # FILL IN YOUR IMPLEMENTATION
    pass

# Expected output: a table and plot showing the trade-off

In [ ]:
# SOLUTION
# --------
filter_sizes = [50, 100, 200]
filter_comparison = []

for filter_k in filter_sizes:
    embedded_k = max(10, min(35, filter_k // 3))

    # Stage 1: filter
    t0 = time.perf_counter()
    X_f, _ = filter_stage(X, y, top_k=filter_k)
    t_f = time.perf_counter() - t0

    # Stage 2: embedded
    t0 = time.perf_counter()
    X_e, _ = embedded_stage_lasso(X_f, y, top_k=embedded_k)
    t_e = time.perf_counter() - t0

    # Stage 3: GA wrapper
    t0 = time.perf_counter()
    final_feats, best_fit, _ = ga_wrapper(X_e, y, pop_size=20, n_generations=20)
    t_ga_stage = time.perf_counter() - t0

    # Evaluate
    final_idx = [list(X.columns).index(f) for f in final_feats]
    cv_scores = cross_val_score(
        GradientBoostingClassifier(n_estimators=100, random_state=42),
        X.values[:, final_idx], y,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='balanced_accuracy'
    )

    total_t = t_f + t_e + t_ga_stage
    filter_comparison.append({
        'filter_k': filter_k,
        'embedded_k': X_e.shape[1],
        'final_k': len(final_feats),
        'total_time': total_t,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
    })
    print(f"filter_k={filter_k}: {len(final_feats)} final features, "
          f"acc={cv_scores.mean():.4f}, time={total_t:.1f}s")

comp_df = pd.DataFrame(filter_comparison)
print("\n", comp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(comp_df['total_time'], comp_df['cv_mean'], 'o-',
         color='#2196F3', linewidth=2, markersize=10)
for _, row in comp_df.iterrows():
    ax.annotate(f"filter_k={int(row['filter_k'])}",
                 xy=(row['total_time'], row['cv_mean']),
                 xytext=(5, 5), textcoords='offset points', fontsize=9)
ax.set_xlabel('Total pipeline time (seconds)', fontsize=11)
ax.set_ylabel('CV Balanced Accuracy', fontsize=11)
ax.set_title('Filter Aggressiveness: Time vs Quality Trade-off', fontsize=11)
plt.tight_layout()
plt.savefig('filter_tradeoff.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

### Key Takeaways

1. **Cascading reduces computation substantially:** Filter (fast) → LASSO (moderate) → GA (expensive but on small space) achieves 5–20× speedup over a direct GA on the full feature space.

2. **The filter stage is the cheapest:** MI scoring takes 1–3 seconds even on 500 features. This is the "free lunch" — a nearly zero-cost noise reduction step.

3. **Quality is not significantly impaired:** The cascade pipeline achieves comparable or better CV accuracy versus the direct full-space GA, because it starts the expensive search from a better candidate set (noise-free features).

4. **Filter aggressiveness matters:** Too aggressive a filter (small top_k) risks dropping informative features. Too conservative (large top_k) reduces the computational savings. Optimal is typically 3–5× the target final feature count.

5. **The GA converges faster from better starting points:** Seeded initialisation from the LASSO-ranked candidates gives the GA a head-start compared to random initialisation on the full feature space.

### What's Next
Notebook 03 builds a meta-learning recommender that automatically decides which selector (or cascade) to use based on dataset properties.

### Additional Resources
- Guide 02: `guides/02_hybrid_methods_guide.md`
- Bolón-Canedo et al. (2015): Recent advances in feature selection for big data — computational analysis of hybrid pipelines